# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib pandas --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import RegularPolygon
import math
from IPython.display import Markdown, display

**Declarations**

Set the parameters below before running the simulation. We use Zurich as playground. With the city of Zurich covering roughly 75 km² and about 500'000 people, the average user density is ≈ 7000 users/km². In an extreme case like the Zurich Street Parade, with an additional 800'000 people (≈ 1.3 million total), the density rises to ≈ 170000 users/km². 

This is a simplified model to provide an introductory understanding. We assume perfect free-space conditions and evenly distributed users, whereas real cellular planning is a lot more complex.

Traffic Parameters
* `user_density` is the number of active users per square kilometer.
* `data_rate_per_user` is the target data rate per user in bits per second.
* `gauss_strength` is the Gaussian concentration strength (0..1). 0 = uniform distribution, 1 = strong central concentration.

Device Parameters
* `frequency` is the carrier frequency in Hz.
* `bandwidth` is the total channel bandwidth in Hz.

In [ ]:
# --- Declarations: editable variables for students ---
number_of_users = ...
data_rate_per_user = ...
gauss_strength = ...

frequency = ...
bandwidth = ...

user_density = number_of_users / 75

In [ ]:
#  --- Calculations & Fixed Data ---

# Receiver parameters
rx_sensitivity = -90
antenna_gain = 3
sinr_target = 10
tx_power = 23

# Physicial constants
speed_of_light = 3e8
noise_power_density = -174


### 1. Coverage: Determining Cell Size
The planning of a cellular network involves balancing coverage (based on power levels) and capacity (based on the number of clients and data requirements). Coverage is primarily determined by the transmit power $P_{\text{TX}}$, the receiver sensitivity $P_{\text{RX, min}}$, and the path loss $L_{\text{path}}$.

#### 1.1 Link Budget
The link budget equation relates the transmit power to the minimum received power:
$$
P_{\text{RX}} = P_{\text{TX}} + G_{\text{TX}} + G_{\text{RX}} - L_{\text{path}},
$$
where
- $P_{\text{RX}}$: Received power at the cell edge (dBm).
- $P_{\text{TX}}$: Transmit power of the base station (dBm).
- $G_{\text{TX}}, G_{\text{RX}}$: Antenna gains of the transmitter and receiver (dBi).
- $L_{\text{path}}$: Path loss (dB), which depends on distance, frequency, and environmental factors.



In [ ]:
# --- Free-space Path Loss ---
fspl = tx_power + 2 * antenna_gain - rx_sensitivity

#### 1.2 Path Loss Model
Path loss can be estimated using models like the Hata or Cost-231 model. A simple free-space path loss (FSPL) is:
$$
L_{\text{path}} = 20\log_{10}(d) + 20\log_{10}(f) + 20\log_{10}\left(\frac{4\pi}{c}\right),
$$
where
- $d$: Distance between transmitter and receiver (meters).
- $f$: Frequency (Hz).
- $c$: Speed of light $3 \times 10^8 \, \text{m/s}$.

Rearranging to solve for $d_{\text{max}}$ (maximum cell radius):
$$
d_{\text{max}} = 10^{\frac{P_{\text{TX}} + G_{\text{TX}} + G_{\text{RX}} - P_{\text{RX, min}} - 20\log_{10}(f) - 20\log_{10}(4\pi/c)}{20}}
$$

In [ ]:
# --- Calculate coverage radius ---
def coverage_radius(tx_power, rx_sensitivity, frequency, antenna_gain):
    fspl = tx_power + 2 * antenna_gain - rx_sensitivity  # Free-space path loss
    wavelength = speed_of_light / frequency
    radius = 10 ** ((fspl - 20 * np.log10(4 * np.pi / wavelength)) / 20)
    return radius

### 2. Capacity: Adjusting Cell Size
Once coverage is determined, capacity considerations can shrink the cell size to accommodate more users or higher data rates.

#### 2.1 Capacity per Cell
The capacity $C$ of a single cell is given by:
$$
C = N \cdot \log_2(1 + \text{SINR}),
$$
where
- $N$: Number of resource blocks or channels available.
- $\text{SINR}$: Signal-to-Interference-and-Noise Ratio, defined as:
  $$
  \text{SINR} = \frac{P_{\text{signal}}}{P_{\text{interference}} + P_{\text{noise}}}
  $$
- $P_{\text{noise}} = k T B$, where $k$: Boltzmann constant, $T$: temperature in Kelvin, $B$: bandwidth.



In [ ]:
# --- Calculate capacity radius ---
sinr_linear = 10 ** (sinr_target / 10)
cell_capacity = bandwidth * np.log2(1 + sinr_linear)

#### 2.2 User Demand
The total demand per cell $D_{\text{cell}}$ is:
$$
D_{\text{cell}} = U \cdot R_{\text{user}},
$$
where
- $U$: Number of users in the cell.
- $R_{\text{user}}$: Average data rate required per user.

To ensure sufficient capacity:
$$
D_{\text{cell}} \leq C
$$



In [ ]:
# --- Calculate user demand ---
total_demand = user_density * data_rate_per_user

#### 2.3 Cell Splitting
If $D_{\text{cell}} > C$, the cell size needs to be reduced. The area of the cell is proportional to $d_{\text{max}}^2$, so reducing the cell size proportionally increases the capacity (more cells in the same area).




In [ ]:
# --- Capacity radius function ---
def capacity_radius(user_density, data_rate_per_user, bandwidth, sinr_linear):
    total_demand = user_density * data_rate_per_user  # bps per km^2
    cell_capacity = bandwidth * np.log2(1 + sinr_linear)  # bps
    if total_demand > 0:
        cell_area = cell_capacity / total_demand  # km^2
    else:
        cell_area = 1e12  # extremely high (no limit)
    radius = np.sqrt(cell_area / np.pi) * 1000  # Convert to meters
    return radius

### 3. Combining Coverage and Capacity
The relationship between coverage and capacity is a trade-off:
1. Coverage-Limited: Large cells with fewer users, where $d_{\text{max}}$ is determined by $P_{\text{TX}}$ and $P_{\text{RX, min}}$.
2. Capacity-Limited: Smaller cells in high-density areas, where $d_{\text{max}}$ is determined by the required capacity per cell.

- Coverage-limited region:
  $$
  d_{\text{coverage}} = f(P_{\text{TX}}, P_{\text{RX, min}}, L_{\text{path}})
  $$
- Capacity-limited region:
  $$
  d_{\text{capacity}} \propto \sqrt{\frac{C}{D_{\text{cell}}}}
  $$

By combining these equations, we can model the cell radius $d_{\text{cell}}$ dynamically as:
$$
d_{\text{cell}} = \min(d_{\text{coverage}}, d_{\text{capacity}})
$$

In [ ]:
# --- Combining Coverage and Capacity ---
d_coverage = coverage_radius(tx_power, rx_sensitivity, frequency, antenna_gain)
d_capacity = capacity_radius(user_density, data_rate_per_user, bandwidth, sinr_linear)
d_cell = min(d_coverage, d_capacity)

## 4. Regular Grid in Zürich

In [ ]:
# --- Visualization: draw hexagonal cells (uniform-aligned, scale-down-only) ---
import matplotlib.image as mpimg
from matplotlib.patches import RegularPolygon, Circle

def draw_hexagonal_cells(
        d_capacity_ref, d_coverage, user_density,
        grid_extent_m=3000, city_diameter_m=5000,
        background_image_path="../../Admin/picture/zurich_map.jpg"
    ):
    cx, cy = 0, 0
    city_R = city_diameter_m / 2

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')

    try:
        img = mpimg.imread(background_image_path)
        extent = [-grid_extent_m, grid_extent_m, -grid_extent_m, grid_extent_m]
        ax.imshow(img, extent=extent, origin='upper', alpha=0.8, zorder=0)
    except Exception as e:
        print(f"[warn] Map load failed: {e}")

    # Hex grid parameters
    uniform_radius = min(d_coverage, d_capacity_ref)
    base_radius = max(1.0, uniform_radius)
    dx = 1.5 * base_radius
    dy = np.sqrt(3) * base_radius

    # Gaussian parameters
    sigma = city_R / 3.0
    gauss_integral = 2 * math.pi * sigma**2 * (1 - math.exp(-city_R**2 / (2 * sigma**2)))
    gauss_avg = gauss_integral / (math.pi * city_R**2)

    # Draw hexagons
    count = 0
    small_count = 0
    i_min = int(-grid_extent_m / dx) - 2
    i_max = int(grid_extent_m / dx) + 2
    j_min = int(-grid_extent_m / dy) - 2
    j_max = int(grid_extent_m / dy) + 2
    for i in range(i_min, i_max + 1):
        for j in range(j_min, j_max + 1):
            x = i * dx
            y = j * dy + (i % 2) * dy / 2
            if (x - cx)**2 + (y - cy)**2 <= city_R**2:
                r = math.hypot(x - cx, y - cy)

                # Gaussian multiplier (normalized)
                G_r = math.exp(- (r**2) / (2 * sigma**2))
                multiplier = (1.0 - gauss_strength) + gauss_strength * (G_r / gauss_avg)
                local_user_density = user_density * multiplier
                local_capacity_radius = capacity_radius(local_user_density, data_rate_per_user, bandwidth, sinr_linear)

                # scale-down-only: never exceed uniform_radius
                this_radius = min(uniform_radius, local_capacity_radius)
                if this_radius < 0.1:
                    this_radius = 0.1

                # count small (scaled-down) cells
                if this_radius < uniform_radius:
                    small_count += 1

                # color selection
                if this_radius < uniform_radius * 0.9:
                    color = 'red'
                elif local_capacity_radius < d_coverage:
                    color = 'green'
                else:
                    color = 'blue'

                hexagon = RegularPolygon(
                    xy=(x, y),
                    numVertices=6,
                    radius=this_radius,
                    orientation=np.radians(30),
                    facecolor=color,
                    alpha=0.45,
                    edgecolor='black',
                    lw=0.5,
                    zorder=2
)
                ax.add_patch(hexagon)
                count += 1

    print(f"  10) Cell Count: {count:,} ({small_count:,} too small)")
    ax.add_patch(Circle((cx, cy), city_R, fill=False, color='white', lw=2, zorder=3))
    ax.set_xlim(-grid_extent_m, grid_extent_m)
    ax.set_ylim(-grid_extent_m, grid_extent_m)
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
    ax.set_title(f"Zurich – uniform-aligned hex grid (green=capacity, blue=coverage), gauss={gauss_strength}")
    ax.grid(True, ls='--', alpha=0.3)
    plt.show()

In [ ]:
# Compute & Display
d_coverage = coverage_radius(tx_power, rx_sensitivity, frequency, antenna_gain)
d_capacity = capacity_radius(user_density, data_rate_per_user, bandwidth, sinr_linear)
d_cell = min(d_coverage, d_capacity)

print(f"Network Summary")
print(f"  1) Users in Zurich: {number_of_users:,.0f}")
print(f"  2) Base User Density ≈ {user_density:,.0f} users/km²")
print(f"  3) Coverage radius: {d_coverage:.1f} m")
print(f"  4) Capacity radius (uniform avg): {d_capacity:.1f} m")
print(f"  5) Gaussian concentration strength: {gauss_strength}")

print(f"\n  6) Transmit power (Tx): {tx_power:.2f} dBm")
print(f"  7) Target data rate per user: {data_rate_per_user/1e6:.2f} Mb/s")
print(f"  8) Channel bandwidth: {bandwidth/1e6:.2f} MHz")
print(f"  9) Carrier frequency: {frequency/1e9:.2f} GHz")

# Draw the grid using per-cell local capacities
draw_hexagonal_cells(d_capacity, d_coverage, user_density)